In [3]:
import pandas as pd
import numpy as np
import json
import re, pytz, os, requests, sys
from pathlib import Path
from datetime import datetime
import sys
sys.path.append("/workspaces/service-data")

from src.clean import clean_percentage, clean_fiscal_yr, normalize_string, standardize_column_names
from src.load import load_csv, download_csv_files
from src.export import export_to_csv
from src.merge import merge_si, merge_ss
from src.utils import dept_list, program_list
from main import get_config

import pandas as pd
import numpy as np
import pytz
from pathlib import Path

base_dir = Path.cwd()
parent_dir = base_dir.parent
config = get_config()
# dept = dept_list(config)


In [ ]:
# download_csv_files(config)
ib_orgs = load_csv('infobase_orgs.csv', config)

ib_orgs = pd.merge(
    ib_orgs.groupby(['id'], as_index=False)['year'].max(),
    ib_orgs[['year', 'id', 'legal_title_en', 'applied_title_en', 'legal_title_fr', 'applied_title_fr']],
    on=['id', 'year']
    )

ib_orgs['department_en'] = ib_orgs.loc[:, 'applied_title_en'].fillna(ib_orgs.loc[:, 'legal_title_en'])
ib_orgs['department_fr'] = ib_orgs.loc[:, 'applied_title_fr'].fillna(ib_orgs.loc[:, 'legal_title_fr'])

ib_orgs=(ib_orgs
            .rename(columns={'id':'org_id'})
            .drop(columns=['year', 'legal_title_en', 'applied_title_en', 'legal_title_fr', 'applied_title_fr'])
            .drop_duplicates()
)

ib_orgs['org_id'] = ib_orgs['org_id'].astype(str)

# ib_orgs.groupby(['org_id']).count()



,department_en,department_fr
org_id,,
1,1,1
10,1,1
100,1,1
103,1,1
104,1,1
...,...,...
95,1,1
96,1,1
97,1,1


In [ ]:
def dept_list(config, export=False):
    """
    Get a list of departments with their English and French names.
    """
    try:
        ifoi_en = load_csv('ifoi_en.csv', config)
        ifoi_fr = load_csv('ifoi_fr.csv', config)

        
        # # Process English names
        # dept_en = ifoi_en.copy()
        # dept_en['department_en'] = dept_en.iloc[:,5].fillna(dept_en.iloc[:,4])
        # dept_en = dept_en[['org_id', 'department_en']]
        
        # # Process French names
        # dept_fr = ifoi_fr.copy()
        # dept_fr['department_fr'] = dept_fr.iloc[:,5].fillna(dept_fr.iloc[:,4])
        # dept_fr = dept_fr[['org_id', 'department_fr']]
        
        # # Merge English and French department names
        # dept = pd.merge(
        #     dept_en,
        #     dept_fr,
        #     on='org_id',
        #     how='outer'
        # )
        
        # Standardize column names and convert org_id to string
        dept = standardize_column_names(dept)
        dept['org_id'] = dept['org_id'].astype(str)

        # Remove any duplicate entries
        dept = dept.drop_duplicates()
        
        if export:
            UTILS_DIR = config['output_dir'] / config['utils_dir']
            export_to_csv(
                data_dict={'dept': dept},
                output_dir=UTILS_DIR,
            )
        
        return dept
    
    except Exception as e:
        logger.error("Error: %s", e, exc_info=True)